# Modele: Classification du Niveau de Risque des Seismes
Ce notebook compare 3 algorithmes de machine learning pour classer le niveau de risque des seismes selon leur magnitude (`mag`) :

- **Faible (0)** : `mag` < 4.0
- **Moyen (1)** : 4.0 <= `mag` < 6.0
- **Eleve (2)** : `mag` >= 6.0

**Algorithmes evalues :**
1. k-Nearest Neighbors (KNN)
2. Arbre de decision (Decision Tree)
3. Forêt aleatoire (Random Forest)

In [1]:
# Importation des bibliotheques requises
import os
import joblib
import skops.io as sio
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.pyfunc
import hashlib
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='mlflow')

# Chargement du dataset
file_path = os.path.join("data", "usgs_earthquakes_2025.csv")

df = pd.read_csv(file_path)
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])

display(df.head())


Nombre de lignes : 29062
Nombre de colonnes : 35


,mag,place,time,updated,tz,url,detail,felt,cdi,mmi,...,title,longitude,latitude,depth,event_id,date_time,year,month,day,hour
0,4.2,"70 km W of Abra Pampa, Argentina",1751241591065,1757018402040,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,"M 4.2 - 70 km W of Abra Pampa, Argentina",-66.3753,-22.6251,247.528,us7000q9mu,2025-06-29 23:59:51.065000+00:00,2025,6,29,23
1,2.5,"1 km E of Nikolski, Alaska",1751241495097,1757018405040,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,"M 2.5 - 1 km E of Nikolski, Alaska",-168.8412,52.9376,61.149,us7000qb5r,2025-06-29 23:58:15.097000+00:00,2025,6,29,23
2,2.8,"31 km SSE of Nikolski, Alaska",1751241219318,1757018405040,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,"M 2.8 - 31 km SSE of Nikolski, Alaska",-168.6544,52.6804,35.000,us7000qb5v,2025-06-29 23:53:39.318000+00:00,2025,6,29,23
3,3.8,"153 km S of Nikolski, Alaska",1751241150341,1757018402040,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,"M 3.8 - 153 km S of Nikolski, Alaska",-168.5438,51.5761,10.000,us7000q9mt,2025-06-29 23:52:30.341000+00:00,2025,6,29,23
4,4.5,"185 km WSW of Tual, Indonesia",1751240317514,1757018402040,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,"M 4.5 - 185 km WSW of Tual, Indonesia",131.2977,-6.4544,35.000,us7000q9mq,2025-06-29 23:38:37.514000+00:00,2025,6,29,23


## 1. Preparation de la Cible (Target) et des Caracteristiques (Features)

In [2]:
# 1. Variable cible (Magnitude)

df = df.loc[df["mag"].notna()].copy()

conditions = [
    df['mag'] < 4.0,
    (df['mag'] >= 4.0) & (df['mag'] < 6.0),
    df['mag'] >= 6.0,
]
choices = [0, 1, 2]  # 0: Faible, 1: Moyen, 2: Eleve
df['risk_level'] = np.select(conditions, choices)

print("\nDistribution de Risk_Level :")
print(df["risk_level"].value_counts(dropna=False))
df_risk2 = df[df['risk_level'] == 2]
print(df_risk2.head())



Distribution de Risk_Level :
risk_level
1    18087
0    10831
2      144
Name: count, dtype: int64
     mag                              place           time        updated  tz  \
123  6.6                         Scotia Sea  1751099540957  1757018401040 NaN   
155  6.0  65 km E of Sarangani, Philippines  1751065630371  1757018400040 NaN   
262  6.2        southern Mid-Atlantic Ridge  1750889773700  1757018399040 NaN   
460  6.2     east of the Philippine Islands  1750730286248  1756848222040 NaN   
617  6.0          82 km SE of Nemuro, Japan  1750540998562  1756848220040 NaN   

                                                   url  \
123  https://earthquake.usgs.gov/earthquakes/eventp...   
155  https://earthquake.usgs.gov/earthquakes/eventp...   
262  https://earthquake.usgs.gov/earthquakes/eventp...   
460  https://earthquake.usgs.gov/earthquakes/eventp...   
617  https://earthquake.usgs.gov/earthquakes/eventp...   

                                                detail  felt  cd

### Justification du Nettoyage et de l'Exclusion des Variables (Feature Selection)

Afin de construire un modele performant, generalisable et exempt de fuite de donnees (*Data Leakage*), plusieurs variables du dataset ont ete exclues lors du choix des caracteristiques ($X$) :

1. **Variables a valeur constante :**
   - `year` (valeur unique "2025") : Une variable sans variance n'apporte aucune information discriminante pour l'apprentissage et ajoute inutilement de la dimensionnalite.

2. **Identifiants uniques (Haute cardinalite) :**
   - `url`, `detail`, `code`, `ids`, `event_id`, `title`, `place` : Ces champs possèdent 100% (ou presque) de valeurs uniques par enregistrement. Les inclure entraînerait un surapprentissage pur (*overfitting*), car le modèle apprendrait par cœur des identifiants au lieu de repérer des motifs généraux.

3. **Variables entierement ou tres fortement manquantes :**
   - `tz` (100% de valeurs manquantes) : Ne contient aucune donnée exploitable.

4. **Variables d'impact post-evenement (Prevention du Data Leakage) :**
   - `felt`, `cdi`, `mmi`, `alert`, `sig`, `status` : Ces informations représentent l'impact ressenti ou mesuré *après* la survenue du séisme (nombre de signalements citoyens, révisions d'experts, alertes émises). Si l'objectif est d'évaluer le risque dès la détection instrumentale, utiliser ces champs constituerait une fuite de données (*Data Leakage*), faussant artificiellement les performances du modèle en conditions réelles.

5. **Exclusion de la variable source (`mag`) :**
   - `mag` a été retirée des variables prédictives car elle a servi directement à construire la variable cible `risk_level`. Sa conservation donnerait un accès direct à la réponse.

---

**Variables conservees pour la modélisation ($X$) :**
- **Coordonnées géophysiques & spatiales :** `latitude`, `longitude`, `depth`
- **Qualité des mesures physiques de l'épicentre :** `dmin`, `gap`, `rms`, `nst`
- **Contexte technique & catégoriel :** `net`, `magType`, `type`

In [3]:
# Sélection des variables d'entrée (Features) et de la cible (Target)
features = [
    'latitude',
    'longitude',
    'depth',
    'dmin',
    'gap',
    'rms',
    'nst',
    'net',
    'magType',
    'type',
]

X = df[features].copy()
y = df['risk_level']

# Les transformations sont ajustées dans chaque fold, sans consulter le jeu de test.
cat_cols = ['net', 'magType', 'type']
num_cols = [column for column in features if column not in cat_cols]

df_trainning = pd.concat([X, y.rename('risk_level')], axis=1)

# Découpage Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("\nDistribution Y_train :")
print(y_train.value_counts())

print("\nDistribution Y_test :")
print(y_test.value_counts())

def make_preprocessor(scale_numeric=False):
    numeric_transformer = SimpleImputer(strategy='median')
    if scale_numeric:
        numeric_transformer = make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
        )

    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_transformer, num_cols),
            (
                'categorical',
                make_pipeline(
                    SimpleImputer(strategy='most_frequent'),
                    OneHotEncoder(handle_unknown='ignore'),
                ),
                cat_cols,
            ),
        ]
    )

print('Preparation des donnees terminee : split 80/20 et preprocesseur defini.')

X_train : (23249, 10)
X_test  : (5813, 10)

Distribution Y_train :
risk_level
1    14469
0     8665
2      115
Name: count, dtype: int64

Distribution Y_test :
risk_level
1    3618
0    2166
2      29
Name: count, dtype: int64
Preparation des donnees terminee : split 80/20 et preprocesseur defini.


## 2. Entrainement et Evaluation des 3 Modeles

In [4]:
MLFLOW_TRACKING_URI = "http://194.238.26.226:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
trusted_types = [
    "numpy.dtype",
    "scipy.sparse._csr.csr_matrix",
    "sklearn.metrics._dist_metrics.EuclideanDistance64",
    "sklearn.neighbors._kd_tree.KDTree",
    "sklearn.tree._tree.Tree",
]

# Resultats du modele champion sur le jeu de test uniquement
results = {}

@mlflow.trace
def evaluate_model(name, y_true, y_pred):
  acc = accuracy_score(y_true, y_pred)
  prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
  rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
  f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
  f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

  results[name] = {
      'Accuracy': acc,
      'Precision (Macro)': prec_macro,
      'Recall (Macro)': rec_macro,
      'F1-Score (Macro)': f1_macro,
      'F1-Score (Weighted)': f1_weighted,
  }

@mlflow.trace
def log_model_to_mlflow(model, model_name):
    mlflow.sklearn.log_model(
        sk_model=model, name=model_name, skops_trusted_types=trusted_types
    )

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = {
    "Accuracy": "accuracy",
    "Precision": make_scorer(precision_score, average="macro", zero_division=0),
    "Recall": make_scorer(recall_score, average="macro", zero_division=0),
    "F1-Score": make_scorer(f1_score, average="macro", zero_division=0),
}
cv_results_summary = {}
models = {}
model_run_ids = {}
model_artifact_paths = {}

@mlflow.trace
def evaluate_cross_validation(model_name, estimator):
    fold_results = cross_validate(
        estimator, X_train, y_train, cv=cv, scoring=scoring
    )
    mean_scores = {}

    print(f"\nValidation croisee - {model_name}")
    for metric in scoring:
        fold_scores = fold_results[f"test_{metric}"]
        mean_score = fold_scores.mean()
        std_score = fold_scores.std(ddof=1)
        mean_scores[metric] = mean_score
        mean_scores[f"{metric} (std)"] = std_score
        print(
            f"{metric.capitalize()} : moyenne = {mean_score:.4f}, "
            f"ecart-type = {std_score:.4f}"
        )
        mlflow.log_metric(f"CV_{metric}", float(mean_score))
        mlflow.log_metric(f"CV_{metric}_std", float(std_score))

    cv_results_summary[model_name] = mean_scores


# ---------------------------------------------------------
# Algorithme 1 : KNN
# ---------------------------------------------------------
experiment_name = "Risk_Zone_Model_2"
mlflow.set_experiment(experiment_name)
experiment = mlflow.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id
mlflow.set_experiment_tag(
    "mlflow.note.content",
    "Expérience de détection de zone de risque."
)
mlflow.set_experiment(experiment_id=experiment_id)

with mlflow.start_run(run_name="KNN", nested=True):
    models["KNN"] = make_pipeline(
        make_preprocessor(scale_numeric=True),
        KNeighborsClassifier(n_neighbors=5, weights='distance'),
    )
    evaluate_cross_validation("KNN", models["KNN"])
    models["KNN"].fit(X_train, y_train)

    # Génération des prédictions pour le modèle KNN
    y_pred = models["KNN"].predict(X_test)

    class_labels = [
        "Faible (0)", 
        "Moyen (1)", 
        "Eleve (2)"
    ]

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred, labels=models["KNN"].classes_)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(cmap="Blues", values_format="d", ax=ax)
    ax.set_title("Matrice de confusion — risque zone sismique")
    plt.tight_layout()
    plt.close(fig)

    mlflow.log_figure(fig, "confusion_matrix.png")

    mlflow.log_artifact(file_path, artifact_path="datasets")
    mlflow.log_param("n_neighbors", 5)
    mlflow.log_param("weights", "distance")
    log_model_to_mlflow(models["KNN"], "Risk_zone_model_KNN")
    model_run_ids["KNN"] = mlflow.active_run().info.run_id
    model_artifact_paths["KNN"] = "Risk_zone_model_KNN"

# ---------------------------------------------------------
# Algorithme 2 : Decision Tree
# ---------------------------------------------------------
with mlflow.start_run(run_name="Decision Tree", nested=True):
    with mlflow.start_span("Decision Tree start Training"):
        print("Début Entrainement du modele Decision Tree...")

    models["Decision Tree"] = make_pipeline(
        make_preprocessor(),
        DecisionTreeClassifier(
            max_depth=10, class_weight='balanced', random_state=42
        ),
    )
    evaluate_cross_validation("Decision Tree", models["Decision Tree"])
    models["Decision Tree"].fit(X_train, y_train)

    # Génération des prédictions pour le modèle KNN
    y_pred = models["Decision Tree"].predict(X_test)

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred, labels=models["Decision Tree"].classes_)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(cmap="Blues", values_format="d", ax=ax)
    ax.set_title("Matrice de confusion — risque zone sismique")
    plt.tight_layout()
    plt.close(fig)

    mlflow.log_figure(fig, "confusion_matrix.png")

    mlflow.log_artifact(file_path, artifact_path="datasets")
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    log_model_to_mlflow(models["Decision Tree"], "Risk_zone_model_DecisionTree")
    model_run_ids["Decision Tree"] = mlflow.active_run().info.run_id
    model_artifact_paths["Decision Tree"] = "Risk_zone_model_DecisionTree"

# ---------------------------------------------------------
# Algorithme 3 : Random Forest
# ---------------------------------------------------------
with mlflow.start_run(run_name="Random Forest", nested=True):
    with mlflow.start_span("Random Forest start Training"):
        print("Début Entrainement du modele Random Forest...")

    models["Random Forest"] = make_pipeline(
        make_preprocessor(),
        RandomForestClassifier(
            n_estimators=150,
            max_depth=12,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
        ),
    )
    evaluate_cross_validation("Random Forest", models["Random Forest"])
    models["Random Forest"].fit(X_train, y_train)

    # Génération des prédictions pour le modèle KNN
    y_pred = models["Random Forest"].predict(X_test)

    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred, labels=models["Random Forest"].classes_)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(cmap="Blues", values_format="d", ax=ax)
    ax.set_title("Matrice de confusion — risque zone sismique")
    plt.tight_layout()
    plt.close(fig)
    mlflow.log_figure(fig, "confusion_matrix.png")
    mlflow.log_artifact(file_path, artifact_path="datasets")
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_jobs", -1)

    dataset = mlflow.data.from_pandas(
        df_trainning,
        source="../data/usgs_earthquakes_2025.csv",
        name="risk_zone_dataset"
    )
    mlflow.log_input(dataset, context="training")
    log_model_to_mlflow(models["Random Forest"], "Risk_zone_model_RandomForest")
    model_run_ids["Random Forest"] = mlflow.active_run().info.run_id
    model_artifact_paths["Random Forest"] = "Risk_zone_model_RandomForest"
#--------------------------------------------------------------
# Le F1 macro moyen de validation croisee choisit le champion.
cv_results_df = pd.DataFrame(cv_results_summary).T
best_model_name = cv_results_df["F1-Score"].idxmax()
best_model = models[best_model_name]
best_model_run_id = model_run_ids[best_model_name]

# Evaluation finale unique du champion sur le jeu de test
best_model_predictions = best_model.predict(X_test)
evaluate_model(best_model_name, y_test, best_model_predictions)
print(f"\nEvaluation finale sur le jeu de test - {best_model_name}")
for metric_name, metric_value in results[best_model_name].items():
    print(f"{metric_name} : {metric_value:.4f}")

mlflow_client = mlflow.MlflowClient()
mlflow_client.set_tag(best_model_run_id, "is_champion", "true")
mlflow_client.set_tag(
    best_model_run_id,
    "mlflow.note.content",
    f"Champion choisi par F1 macro CV: {best_model_name}",
)
for metric_name, metric_value in results[best_model_name].items():
    metric_key = metric_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    mlflow_client.log_metric(
        best_model_run_id, f"TEST_{metric_key}", float(metric_value)
    )

mlflow.register_model(
    model_uri=f"runs:/{best_model_run_id}/{model_artifact_paths[best_model_name]}",
    name="Risk_Zone_Champion_Model",
)
print(f"Modele champion enregistre : {best_model_name}")
pd.DataFrame(cv_results_summary).T.round(4).sort_values(
    by="F1-Score", ascending=False
)


Validation croisee - KNN
Accuracy : moyenne = 0.9787, ecart-type = 0.0027
Precision : moyenne = 0.7522, ecart-type = 0.0778
Recall : moyenne = 0.6896, ecart-type = 0.0276
F1-score : moyenne = 0.7046, ecart-type = 0.0388
🏃 View run KNN at: http://194.238.26.226:5000/#/experiments/139/runs/02292183e1ab403d9d13fcaea074f97c
🧪 View experiment at: http://194.238.26.226:5000/#/experiments/139
Début Entrainement du modele Decision Tree...

Validation croisee - Decision Tree
Accuracy : moyenne = 0.9619, ecart-type = 0.0047
Precision : moyenne = 0.7036, ecart-type = 0.0076
Recall : moyenne = 0.8739, ecart-type = 0.0374
F1-score : moyenne = 0.7315, ecart-type = 0.0117
🏃 View run Decision Tree at: http://194.238.26.226:5000/#/experiments/139/runs/78bd97f762be46f3b31510f9c6fafcbe
🧪 View experiment at: http://194.238.26.226:5000/#/experiments/139
Début Entrainement du modele Random Forest...

Validation croisee - Random Forest
Accuracy : moyenne = 0.9647, ecart-type = 0.0041
Precision : moyenne = 0

Registered model 'Risk_Zone_Champion_Model' already exists. Creating a new version of this model...
2026/10/06 19:51:33 WARNING mlflow.tracking._model_registry.fluent: Run with id 126231cdefd342eab583db5b7532ce9f has no artifacts at artifact path 'Risk_zone_model_RandomForest', registering model based on models:/m-7983161fcb4d4c56919be89224fb49d1 instead
2026/10/06 19:51:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Risk_Zone_Champion_Model, version 21
Created version '21' of model 'Risk_Zone_Champion_Model'.


Modele champion enregistre : Random Forest


,Accuracy,Accuracy (std),Precision,Precision (std),Recall,Recall (std),F1-Score,F1-Score (std)
Random Forest,0.9647,0.0041,0.7111,0.0092,0.9092,0.0409,0.7424,0.0144
Decision Tree,0.9619,0.0047,0.7036,0.0076,0.8739,0.0374,0.7315,0.0117
KNN,0.9787,0.0027,0.7522,0.0778,0.6896,0.0276,0.7046,0.0388


[Trace(trace_id=tr-68ea595dd49e349d3ab899f4bcbe44b2), Trace(trace_id=tr-07706cf76fa05beb9bc5475bcd5bbf96), Trace(trace_id=tr-48145a814e845bf4e823b0b7ca4d8bc5), Trace(trace_id=tr-8b76c26430d3bea66599d386b2e95188), Trace(trace_id=tr-75993d2208824949787c2eb30739d535), Trace(trace_id=tr-1cb2b7e6eec5e48637fc0a952fd173e3), Trace(trace_id=tr-668bbc10fde9726696a39aa34d56bd58)]

## 3. Rapport de Classification Detailled (Print par Modèle)

In [5]:
target_names = ['Faible (<4.0)', 'Moyen (4.0-5.9)', 'Élevé (>=6.0)']

print(f"=================== MODELE CHAMPION : {best_model_name} ===================")
print(classification_report(
    y_test,
    best_model_predictions,
    target_names=target_names,
    zero_division=0,
))

=================== MODELE CHAMPION : Random Forest ===================
                 precision    recall  f1-score   support

  Faible (<4.0)       0.98      0.98      0.98      2166
Moyen (4.0-5.9)       0.99      0.95      0.97      3618
  Élevé (>=6.0)       0.15      0.79      0.26        29

       accuracy                           0.96      5813
      macro avg       0.71      0.91      0.74      5813
   weighted avg       0.98      0.96      0.97      5813



## 4. Tableau Comparatif des Performances (Moyenne et Ecart-type)

In [6]:
cv_results_df.round(4).sort_values(by='F1-Score', ascending=False)

,Accuracy,Accuracy (std),Precision,Precision (std),Recall,Recall (std),F1-Score,F1-Score (std)
Random Forest,0.9647,0.0041,0.7111,0.0092,0.9092,0.0409,0.7424,0.0144
Decision Tree,0.9619,0.0047,0.7036,0.0076,0.8739,0.0374,0.7315,0.0117
KNN,0.9787,0.0027,0.7522,0.0778,0.6896,0.0276,0.7046,0.0388


## 5. Analyse Comparative et Choix du Meilleur Modele

### Critère de sélection
Le modèle champion est choisi selon le **F1-Score macro moyen de la validation croisée**, calculée sur le jeu d'entraînement. Cette métrique accorde le même poids à chaque classe et convient au déséquilibre présent dans les données.

Pour chaque métrique, le tableau présente la moyenne et l'écart-type entre les folds. À performances moyennes proches, un écart-type plus faible indique des résultats plus stables d'un fold à l'autre.

### Lecture des résultats

1. **KNN :** méthode simple, mais sensible à l'échelle des variables. Son pipeline standardise les variables numériques et encode les catégories en one-hot, sans standardiser les colonnes catégorielles.
2. **Decision Tree :** modèle interprétable, mais susceptible au surapprentissage.
3. **Random Forest :** modèle d'ensemble qui réduit généralement la variance d'un arbre unique.

Aucun algorithme n'est déclaré meilleur à l'avance : le champion dépend des scores de cross-validation observés dans le tableau comparatif.

### Évaluation finale
Après cette sélection, le modèle champion est évalué une seule fois sur le jeu de test indépendant (20 %). Ces métriques estiment sa performance finale; elles ne servent pas à choisir le champion.

In [7]:
# Sauvegarder le modele selectionne par le F1 macro de la cross-validation
output_dir = "../models"
os.makedirs(output_dir, exist_ok=True)


# Extension modifiée en .skops
output_filename = (
    f"Risk_Zone_{best_model_name.replace(' ', '_')}_champion_model.skops"
)

# Remplacement de joblib.dump par sio.dump
sio.dump(best_model, os.path.join(output_dir, output_filename))


# Artefacts du modèle entraîné format skops
mlflow.log_artifact(  
        os.path.join(output_dir, output_filename),
        artifact_path="model.skops",
        run_id=best_model_run_id

)


output_filename = (
    f"Risk_Zone_{best_model_name.replace(' ', '_')}_champion_model.joblib"
)
joblib.dump(best_model, os.path.join(output_dir, output_filename))

   # Artefacts du modèle entraîné format joblib
mlflow.log_artifact(  
        os.path.join(output_dir, output_filename),
        artifact_path="model",
        run_id=best_model_run_id

)

print(f"Modèle sauvegardé avec succès : {best_model_name}")

Modèle sauvegardé avec succès : Random Forest


[Trace(trace_id=tr-1a74e5876153f0ff0dd4335f49dc809c), Trace(trace_id=tr-2186e11fee88bbad6018e3019f7c5438)]